In [1]:
import pandas as pd
import re
import zstandard as zstd
import io
import orjson
from collections import defaultdict
import json
from tqdm import tqdm
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
import matplotlib.pyplot as plt
import csv
import polars as pl

In [9]:
march_test = pl.scan_parquet('parquet/comments_2024_march_full.parquet')
march_test.collect().head(10)

author,subreddit,count_comments
str,str,i64
"""mouseat9""","""aitah""",1
"""LargeBarnacle7711""","""dnd""",1
"""sgh616""","""foodporn""",1
"""Onlyroad4adrifter""","""facepalm""",1
"""ewejoser""","""worldnews""",1
"""guest_username2""","""peterexplainsthejoke""",1
"""learnyouathang""","""askreddit""",1
"""HonestBalloon""","""politics""",1
"""Synyster182""","""unpopularopinion""",1


In [2]:
# load march comments into polar df

df_mc = pl.scan_csv("comments_2024_march_full.csv")
df_mc.collect()

author,subreddit,count_comments
str,str,i64
"""mouseat9""","""aitah""",1
"""LargeBarnacle7711""","""dnd""",1
"""sgh616""","""foodporn""",1
"""Onlyroad4adrifter""","""facepalm""",1
"""ewejoser""","""worldnews""",1
…,…,…
"""Prowindowlicker""","""atheism""",1
"""Time-Ad-7055""","""facepalm""",1
"""wholeearthmama""","""aww""",1


In [3]:
# aggregate march comments

df_mc_agg = df_mc.group_by(['author', 'subreddit']).agg(
    pl.col("count_comments").sum().alias('count_comments')).sort('author')

df_mc_agg.collect()

author,subreddit,count_comments
str,str,i64
"""------------------16""","""characterai""",5
"""------------------GL""","""gaming""",1
"""------------------GL""","""personalfinance""",2
"""------------------GL""","""mildlyinteresting""",9
"""------------------GL""","""beamazed""",1
…,…,…
"""zzzzzzzzzzzzvzzzzvzz""","""startups""",3
"""zzzzzzzzzzzzzzzz0""","""trueoffmychest""",2
"""zzzzzzzzzzzzzzzz0""","""college""",9


In [4]:
# load march sub into polar df
df_ms = pl.scan_csv("submissions_2024_march_full.csv")
df_ms.collect()

author,subreddit,count_posts
str,str,i64
"""IntroductionFit4364""","""coffee""",1
"""mattthheww""","""indieheads""",1
"""3kOlen""","""popheads""",1
"""Key-Tie1861""","""teenagers""",1
"""WillamThunderfuck""","""popheads""",1
…,…,…
"""barki26""","""finance""",1
"""Ashell77""","""askreddit""",1
"""Live-Ad-6279""","""houseofthedragon""",1


In [5]:
# aggregate march submissions

df_ms_agg = df_ms.group_by(['author', 'subreddit']).agg(
    pl.col("count_posts").sum().alias('count_posts')).sort('author')

df_ms_agg.collect()

author,subreddit,count_posts
str,str,i64
"""---------00---------""","""askreddit""",3
"""---------00---------""","""unexpected""",1
"""--------51""","""youtube""",1
"""-------7654321""","""nostupidquestions""",1
"""-------Tom---------""","""askreddit""",1
…,…,…
"""zzzzzzzzzra""","""nostupidquestions""",1
"""zzzzzzzzzra""","""books""",1
"""zzzzzzzzzra""","""astronomy""",1


In [6]:
# merge aggregated march comments and posts

# merge aggregated march submissions and comments 
march_merged = df_mc_agg.join(
    df_ms_agg,
    on=["author", "subreddit"],
    how="full",
    coalesce=True
).sort('author')

# fill nans for count comments and posts
march_merged = march_merged.with_columns(pl.col('count_comments').fill_null(strategy="zero"),
                                         pl.col('count_posts').fill_null(strategy="zero"))

# show total interaction for march (comment + sub)
march_merged = march_merged.with_columns((pl.col('count_comments') + pl.col('count_posts')).alias('total_interactions'))

# show
march_merged.collect()

author,subreddit,count_comments,count_posts,total_interactions
str,str,i64,i64,i64
"""------------------16""","""characterai""",5,0,5
"""------------------GL""","""pics""",1,0,1
"""------------------GL""","""personalfinance""",2,0,2
"""------------------GL""","""mildlyinteresting""",9,0,9
"""------------------GL""","""entertainment""",1,0,1
…,…,…,…,…
"""zzzzzzzzzzzzvzzzzvzz""","""startups""",3,0,3
"""zzzzzzzzzzzzzzzz0""","""trueoffmychest""",2,0,2
"""zzzzzzzzzzzzzzzz0""","""dating""",5,0,5
